# 03A — Reference Log-Mel Frontend

**Status at start:** PROPOSED / NOT YET FROZEN

This notebook implements and verifies the **HumBugDB/VGGish-style Log-Mel frontend** for the current Example Grid v2.

It does **not** train a classifier.

The purpose of 03A is:

```text
Example Grid v2
    ↓
extract 15,600-sample reference context
    ↓
exact VGGish-style waveform framing
    ↓
STFT magnitude
    ↓
64-bin Mel projection
    ↓
log(mel + 0.01)
    ↓
96 × 64 feature tensor
    ↓
numerical parity check against pinned upstream implementation
```

## Reference frontend contract

- sample rate: **16,000 Hz**
- context: **15,600 samples = 0.975 s**
- waveform frame: **400 samples = 25 ms**
- frame hop: **160 samples = 10 ms**
- complete frames: **96**
- periodic Hann window
- FFT length: **512**, applied *after* framing 400 samples
- spectrum: unnormalized magnitude `abs(rfft(...))`
- Mel filters: **64**
- Mel range: **125–7500 Hz**
- VGGish/HTK-style Mel mapping
- DC Mel weights forced to zero
- no Mel area normalization
- output: natural `log(mel + 0.01)`

`librosa.stft(...)` is deliberately **not** used as the reference implementation here because frame construction must match the upstream VGGish semantics explicitly.

## 3A.1 — Project setup and current artifacts

This section imports the required libraries, discovers the project root without a hard-coded Windows path, resolves the **current Example Grid v2** and frozen split, then authenticates that the manifest still belongs to the accepted 1,898-clip / 1,189-source experiment.

The notebook prefers the reorganized repository layout but can still read the previous flat `data/` layout during migration.

In [1]:
from pathlib import Path
import hashlib
import importlib.util
import json
import platform
import sys
import wave

import numpy as np
import pandas as pd
import scipy
from scipy.io import wavfile
from scipy import signal

import matplotlib
import matplotlib.pyplot as plt

def find_project_root(start=None):
    """
    Locate the repository root without hard-coding D:\\... paths.
    Works from notebooks/ or the old code_by_me/notebooks/ location.
    """
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()

    candidates = [start, *start.parents]

    for p in candidates:
        if (p / "data").exists() and (
            (p / "notebooks").exists()
            or (p / "code_by_me").exists()
            or (p / "third_party").exists()
        ):
            return p

    raise FileNotFoundError(
        "Could not discover project root. "
        "Run this notebook from inside the Edge AI repository."
    )

PROJECT = find_project_root()

print("PROJECT:", PROJECT)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)

PROJECT: D:\VGU-27\Edge AI Project
Python: 3.11.9
NumPy: 2.4.6
SciPy: 1.17.1
pandas: 3.0.6
matplotlib: 3.11.2


### Resolve current data paths

Locate `data/audio/`, the frozen split, and `core_example_grid_v2_stride15360_ref15600_candidates.csv`. The superseded 15,360-context manifest is never used as a fallback.

In [2]:
def first_existing(paths, description):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError(
        f"Could not find {description}.\nChecked:\n"
        + "\n".join(str(Path(p)) for p in paths)
    )

AUDIO_DIR = PROJECT / "data" / "audio"

MANIFEST_PATH = first_existing(
    [
        PROJECT / "data" / "manifests" / "current"
        / "core_example_grid_v2_stride15360_ref15600_candidates.csv",
        PROJECT / "data"
        / "core_example_grid_v2_stride15360_ref15600_candidates.csv",
    ],
    "current Example Grid v2 manifest",
)

SPLIT_PATH = first_existing(
    [
        PROJECT / "data" / "manifests" / "current"
        / "core_single_4species_frozen_split.csv",
        PROJECT / "data" / "core_single_4species_frozen_split.csv",
    ],
    "frozen split",
)

assert AUDIO_DIR.exists(), f"Missing audio directory: {AUDIO_DIR}"

print("Audio:", AUDIO_DIR)
print("Manifest:", MANIFEST_PATH)
print("Frozen split:", SPLIT_PATH)

Audio: D:\VGU-27\Edge AI Project\data\audio
Manifest: D:\VGU-27\Edge AI Project\data\core_example_grid_v2_stride15360_ref15600_candidates.csv
Frozen split: D:\VGU-27\Edge AI Project\data\core_single_4species_frozen_split.csv


### Authenticate Example Grid v2

Verify the expected 32,645 examples, unique identities, fixed stride/context geometry, exact parent metadata, and zero source leakage before any feature extraction.

In [3]:
examples = pd.read_csv(MANIFEST_PATH)
frozen_split = pd.read_csv(SPLIT_PATH)

required_manifest_cols = {
    "example_uid",
    "id",
    "name",
    "species",
    "split",
    "example_index",
    "start_sample_16k",
    "example_stride_samples",
    "reference_context_samples",
    "reference_end_sample_16k",
}

missing = required_manifest_cols - set(examples.columns)
assert not missing, f"Manifest missing columns: {sorted(missing)}"

assert len(examples) == 32645, (
    f"Expected current Example Grid v2 (32645 examples), got {len(examples)}"
)

assert examples["example_uid"].is_unique
assert not examples.duplicated(["id", "start_sample_16k"]).any()
assert (examples["example_stride_samples"] == 15360).all()
assert (examples["reference_context_samples"] == 15600).all()

# Exact parent metadata must agree with canonical frozen split.
parent = (
    examples[["id", "name", "species", "split"]]
    .drop_duplicates()
    .sort_values("id")
    .reset_index(drop=True)
)

frozen_parent = (
    frozen_split[["id", "name", "species", "split"]]
    .sort_values("id")
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    parent,
    frozen_parent,
    check_dtype=False,
)

assert examples.groupby("name")["split"].nunique().max() == 1

print("Examples:", len(examples))
print("Parent clips:", examples["id"].nunique())
print("Sources:", examples["name"].nunique())
print("Source leakage: 0")
print("Example Grid v2 authentication: PASS")

Examples: 32645
Parent clips: 1898
Sources: 1189
Source leakage: 0
Example Grid v2 authentication: PASS


## 3A.2 — Freeze the *candidate* frontend constants locally

These constants reproduce the VGGish-style frontend contract to be tested in this notebook.

They are not considered scientifically frozen until the parity checks below pass.

In [4]:
SAMPLE_RATE = 16000
CONTEXT_SAMPLES = 15600

FRAME_LENGTH = 400
FRAME_HOP = 160
FFT_LENGTH = 512

NUM_MEL_BINS = 64
MEL_MIN_HZ = 125.0
MEL_MAX_HZ = 7500.0
LOG_OFFSET = 0.01

EXPECTED_FRAMES = 96
EXPECTED_SHAPE = (EXPECTED_FRAMES, NUM_MEL_BINS)

def complete_frame_count(n_samples, frame_length, frame_hop):
    if n_samples < frame_length:
        return 0
    return 1 + (n_samples - frame_length) // frame_hop

assert complete_frame_count(15360, FRAME_LENGTH, FRAME_HOP) == 94
assert complete_frame_count(15600, FRAME_LENGTH, FRAME_HOP) == 96
assert complete_frame_count(16000, FRAME_LENGTH, FRAME_HOP) == 98

print("15360 samples ->", complete_frame_count(15360, 400, 160), "frames")
print("15600 samples ->", complete_frame_count(15600, 400, 160), "frames")
print("16000 samples ->", complete_frame_count(16000, 400, 160), "frames")
print("Reference geometry: PASS")

15360 samples -> 94 frames
15600 samples -> 96 frames
16000 samples -> 98 frames
Reference geometry: PASS


## 3A.3 — Audio loading and whole-clip resampling

Core WAV contract:

- native rate = 44.1 kHz
- mono
- PCM24 (3 bytes/sample)
- SciPy decodes PCM24 into left-justified `int32`

Conversion:

```text
PCM24 left-justified int32
    ↓ divide by 2^31
float waveform
    ↓ scipy.signal.resample_poly(up=160, down=441)
16 kHz waveform
```

No peak normalization, RMS normalization, denoising, or signal cleaning is applied.

In [5]:
NATIVE_SAMPLE_RATE = 44100
RESAMPLE_UP = 160
RESAMPLE_DOWN = 441

def load_core_wav_16k(clip_id):
    path = AUDIO_DIR / f"{int(clip_id)}.wav"

    if not path.exists():
        raise FileNotFoundError(path)

    # Physical WAV-header checks.
    with wave.open(str(path), "rb") as wf:
        channels = wf.getnchannels()
        sample_width = wf.getsampwidth()
        native_sr = wf.getframerate()

    if channels != 1:
        raise ValueError(f"{path.name}: expected mono, got {channels} channels")
    if sample_width != 3:
        raise ValueError(
            f"{path.name}: expected PCM24 (3 bytes/sample), got {sample_width}"
        )
    if native_sr != NATIVE_SAMPLE_RATE:
        raise ValueError(
            f"{path.name}: expected {NATIVE_SAMPLE_RATE} Hz, got {native_sr}"
        )

    sr, raw = wavfile.read(path)

    if sr != NATIVE_SAMPLE_RATE:
        raise ValueError(f"{path.name}: scipy sample rate mismatch: {sr}")
    if raw.ndim != 1:
        raise ValueError(f"{path.name}: scipy decoded non-mono array {raw.shape}")
    if raw.dtype != np.int32:
        raise TypeError(
            f"{path.name}: expected left-justified PCM24 int32, got {raw.dtype}"
        )

    x = raw.astype(np.float64) / float(2**31)

    if not np.isfinite(x).all():
        raise ValueError(f"{path.name}: non-finite native waveform")

    y = signal.resample_poly(
        x,
        up=RESAMPLE_UP,
        down=RESAMPLE_DOWN,
    )

    y = np.asarray(y, dtype=np.float64)

    if not np.isfinite(y).all():
        raise ValueError(f"{path.name}: non-finite resampled waveform")

    return y


def extract_reference_context(row, waveform_16k=None):
    if waveform_16k is None:
        waveform_16k = load_core_wav_16k(row["id"])

    start = int(row["start_sample_16k"])
    end = start + CONTEXT_SAMPLES

    if start < 0 or end > len(waveform_16k):
        raise ValueError(
            f"Invalid reference bounds for {row['example_uid']}: "
            f"[{start}, {end}) with waveform length {len(waveform_16k)}"
        )

    context = waveform_16k[start:end]

    if len(context) != CONTEXT_SAMPLES:
        raise AssertionError(
            f"{row['example_uid']}: got {len(context)} samples"
        )
    if not np.isfinite(context).all():
        raise ValueError(f"{row['example_uid']}: non-finite context")

    return context

## 3A.4 — Exact VGGish-style NumPy frontend

Important implementation detail:

**Frame 400 waveform samples first, then zero-pad each FFT to 512.**

We therefore construct waveform frames explicitly instead of asking an STFT library to decide frame geometry.

In [6]:
_MEL_BREAK_FREQUENCY_HERTZ = 700.0
_MEL_HIGH_FREQUENCY_Q = 1127.0

def frame_complete(data, window_length, hop_length):
    """
    Successive complete frames only. No centered padding and no
    incomplete terminal frame.
    """
    data = np.asarray(data)

    if data.ndim != 1:
        raise ValueError(f"Expected 1-D waveform, got {data.shape}")
    if len(data) < window_length:
        return np.empty((0, window_length), dtype=data.dtype)

    n_frames = 1 + (len(data) - window_length) // hop_length

    shape = (n_frames, window_length)
    strides = (data.strides[0] * hop_length, data.strides[0])

    return np.lib.stride_tricks.as_strided(
        data,
        shape=shape,
        strides=strides,
        writeable=False,
    )


def periodic_hann(window_length):
    """
    Periodic Hann used by the VGGish frontend.
    Equivalent to np.hanning(window_length + 1)[:-1].
    """
    return np.hanning(window_length + 1)[:-1]


def stft_magnitude_vggish(
    signal_1d,
    frame_length=FRAME_LENGTH,
    frame_hop=FRAME_HOP,
    fft_length=FFT_LENGTH,
):
    frames = frame_complete(
        np.asarray(signal_1d, dtype=np.float64),
        window_length=frame_length,
        hop_length=frame_hop,
    )

    window = periodic_hann(frame_length)
    windowed = frames * window[np.newaxis, :]

    # np.fft.rfft pads 400-sample input to n=512 here.
    return np.abs(
        np.fft.rfft(
            windowed,
            n=fft_length,
            axis=1,
        )
    )


def hertz_to_mel_vggish(frequencies_hz):
    frequencies_hz = np.asarray(frequencies_hz, dtype=np.float64)
    return (
        _MEL_HIGH_FREQUENCY_Q
        * np.log1p(frequencies_hz / _MEL_BREAK_FREQUENCY_HERTZ)
    )


def spectrogram_to_mel_matrix_vggish(
    num_mel_bins=NUM_MEL_BINS,
    num_spectrogram_bins=FFT_LENGTH // 2 + 1,
    audio_sample_rate=SAMPLE_RATE,
    lower_edge_hertz=MEL_MIN_HZ,
    upper_edge_hertz=MEL_MAX_HZ,
):
    nyquist_hz = audio_sample_rate / 2.0

    if lower_edge_hertz < 0.0:
        raise ValueError("lower_edge_hertz must be >= 0")
    if lower_edge_hertz >= upper_edge_hertz:
        raise ValueError("lower_edge_hertz must be < upper_edge_hertz")
    if upper_edge_hertz > nyquist_hz:
        raise ValueError("upper_edge_hertz cannot exceed Nyquist")

    spectrogram_hz = np.linspace(
        0.0,
        nyquist_hz,
        num_spectrogram_bins,
    )
    spectrogram_mel = hertz_to_mel_vggish(spectrogram_hz)

    band_edges_mel = np.linspace(
        hertz_to_mel_vggish(lower_edge_hertz),
        hertz_to_mel_vggish(upper_edge_hertz),
        num_mel_bins + 2,
    )

    mel_weights = np.empty(
        (num_spectrogram_bins, num_mel_bins),
        dtype=np.float64,
    )

    for i in range(num_mel_bins):
        lower = band_edges_mel[i]
        center = band_edges_mel[i + 1]
        upper = band_edges_mel[i + 2]

        lower_slope = (spectrogram_mel - lower) / (center - lower)
        upper_slope = (upper - spectrogram_mel) / (upper - center)

        mel_weights[:, i] = np.maximum(
            0.0,
            np.minimum(lower_slope, upper_slope),
        )

    # Match VGGish convention: DC bin contributes to no Mel band.
    mel_weights[0, :] = 0.0

    return mel_weights


MEL_MATRIX = spectrogram_to_mel_matrix_vggish()


def logmel_reference(context_16k):
    """
    One 15,600-sample context -> one 96 x 64 Log-Mel tensor.
    """
    x = np.asarray(context_16k, dtype=np.float64)

    if x.ndim != 1:
        raise ValueError(f"Expected 1-D context, got {x.shape}")
    if len(x) != CONTEXT_SAMPLES:
        raise ValueError(
            f"Expected {CONTEXT_SAMPLES} samples, got {len(x)}"
        )
    if not np.isfinite(x).all():
        raise ValueError("Context contains NaN/Inf")

    magnitude = stft_magnitude_vggish(x)

    if magnitude.shape != (EXPECTED_FRAMES, FFT_LENGTH // 2 + 1):
        raise AssertionError(
            f"Unexpected magnitude shape: {magnitude.shape}"
        )

    mel = magnitude @ MEL_MATRIX
    logmel = np.log(mel + LOG_OFFSET)

    if logmel.shape != EXPECTED_SHAPE:
        raise AssertionError(f"Unexpected Log-Mel shape: {logmel.shape}")
    if not np.isfinite(logmel).all():
        raise ValueError("Log-Mel contains NaN/Inf")

    return logmel

## 3A.5 — Deterministic unit tests

These tests check geometry and numerical sanity independently of the mosquito dataset.

In [7]:
rng = np.random.default_rng(12345)

synthetic_inputs = {
    "zeros": np.zeros(CONTEXT_SAMPLES, dtype=np.float64),
    "impulse": np.r_[1.0, np.zeros(CONTEXT_SAMPLES - 1)],
    "440Hz_sine": 0.5 * np.sin(
        2 * np.pi * 440.0
        * np.arange(CONTEXT_SAMPLES, dtype=np.float64)
        / SAMPLE_RATE
    ),
    "noise": rng.normal(0.0, 0.05, CONTEXT_SAMPLES),
}

for label, x in synthetic_inputs.items():
    feat = logmel_reference(x)

    assert feat.shape == EXPECTED_SHAPE
    assert np.isfinite(feat).all()

    print(
        f"{label:12s}",
        "shape =", feat.shape,
        "min =", f"{feat.min():.6f}",
        "max =", f"{feat.max():.6f}",
    )

# Zero input is exactly log(0.01).
zero_expected = np.log(LOG_OFFSET)
zero_actual = logmel_reference(synthetic_inputs["zeros"])

assert np.allclose(
    zero_actual,
    zero_expected,
    rtol=0,
    atol=1e-12,
)

assert MEL_MATRIX.shape == (257, 64)
assert np.all(MEL_MATRIX[0, :] == 0.0)

print("\nSynthetic frontend tests: PASS")

zeros        shape = (96, 64) min = -4.605170 max = -4.605170
impulse      shape = (96, 64) min = -4.605170 max = -4.605170
440Hz_sine   shape = (96, 64) min = -4.604327 max = 4.089175
noise        shape = (96, 64) min = -3.087933 max = 2.131165

Synthetic frontend tests: PASS


## 3A.6 — Numerical parity against pinned upstream `mel_features.py`

This is the critical verification step.

The notebook searches the repository for a local pinned HumBugDB/VGGish `mel_features.py`. It does **not** download or silently substitute a library implementation.

If automatic discovery finds the wrong file or finds nothing, set `UPSTREAM_MEL_FEATURES_PATH` manually to the pinned source used by this project.

### Locate the pinned upstream implementation

Search only local project/reference locations for the pinned HumBugDB/VGGish `mel_features.py`. If it cannot be located, parity verification stops rather than silently substituting another implementation.

In [8]:
UPSTREAM_MEL_FEATURES_PATH = None
REQUIRE_UPSTREAM_PARITY = True

def looks_like_vggish_mel_features(path):
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return False

    markers = [
        "def log_mel_spectrogram",
        "def spectrogram_to_mel_matrix",
        "def stft_magnitude",
        "def frame",
    ]

    return all(marker in text for marker in markers)

if UPSTREAM_MEL_FEATURES_PATH is None:
    preferred_roots = [
        PROJECT / "reference" / "humbugdb",
        PROJECT / "reference",
        PROJECT / "baseline_readiness",
        PROJECT / "archive" / "old_experiments" / "baseline_readiness",
        PROJECT / "review_support",
        PROJECT / "tools" / "review_support",
    ]

    candidates = []

    for root in preferred_roots:
        if not root.exists():
            continue

        for p in root.rglob("mel_features.py"):
            if looks_like_vggish_mel_features(p):
                candidates.append(p)

    # Prefer paths containing humbug/vggish.
    candidates = sorted(
        set(candidates),
        key=lambda p: (
            "humbug" not in str(p).lower(),
            "vggish" not in str(p).lower(),
            len(str(p)),
        ),
    )

    if candidates:
        UPSTREAM_MEL_FEATURES_PATH = candidates[0]

if UPSTREAM_MEL_FEATURES_PATH is None:
    message = (
        "Pinned upstream mel_features.py was not found automatically.\n"
        "Set UPSTREAM_MEL_FEATURES_PATH to the project's pinned "
        "HumBugDB/VGGish mel_features.py before freezing 03A."
    )

    if REQUIRE_UPSTREAM_PARITY:
        raise FileNotFoundError(message)
    else:
        print("WARNING:", message)
else:
    UPSTREAM_MEL_FEATURES_PATH = Path(UPSTREAM_MEL_FEATURES_PATH)
    print("Pinned upstream frontend:", UPSTREAM_MEL_FEATURES_PATH)

FileNotFoundError: Pinned upstream mel_features.py was not found automatically.
Set UPSTREAM_MEL_FEATURES_PATH to the project's pinned HumBugDB/VGGish mel_features.py before freezing 03A.

### Import and fingerprint the pinned source

Load the upstream file read-only and record its SHA-256 so the exact reference implementation used for parity is identifiable.

In [ ]:
upstream = None
upstream_sha256 = None

if UPSTREAM_MEL_FEATURES_PATH is not None:
    upstream_sha256 = hashlib.sha256(
        UPSTREAM_MEL_FEATURES_PATH.read_bytes()
    ).hexdigest()

    spec = importlib.util.spec_from_file_location(
        "pinned_vggish_mel_features",
        UPSTREAM_MEL_FEATURES_PATH,
    )

    upstream = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(upstream)

    print("Pinned source SHA-256:", upstream_sha256)
    print("Upstream module imported: PASS")

### Synthetic numerical parity

Run the same deterministic synthetic signals through both implementations and require near-identical `96 × 64` outputs.

In [ ]:
SYNTHETIC_PARITY_ATOL = 1e-10
synthetic_parity_results = []

if upstream is None:
    if REQUIRE_UPSTREAM_PARITY:
        raise RuntimeError("Upstream module is required for parity.")
else:
    for label, x in synthetic_inputs.items():
        ours = logmel_reference(x)

        theirs = upstream.log_mel_spectrogram(
            x,
            audio_sample_rate=SAMPLE_RATE,
            log_offset=LOG_OFFSET,
            window_length_secs=FRAME_LENGTH / SAMPLE_RATE,
            hop_length_secs=FRAME_HOP / SAMPLE_RATE,
            num_mel_bins=NUM_MEL_BINS,
            lower_edge_hertz=MEL_MIN_HZ,
            upper_edge_hertz=MEL_MAX_HZ,
        )

        theirs = np.asarray(theirs, dtype=np.float64)

        assert theirs.shape == EXPECTED_SHAPE

        max_abs = float(np.max(np.abs(ours - theirs)))
        mean_abs = float(np.mean(np.abs(ours - theirs)))

        synthetic_parity_results.append({
            "input": label,
            "max_abs_diff": max_abs,
            "mean_abs_diff": mean_abs,
        })

        print(
            f"{label:12s}",
            f"max_abs_diff={max_abs:.3e}",
            f"mean_abs_diff={mean_abs:.3e}",
        )

        assert np.allclose(
            ours,
            theirs,
            rtol=0,
            atol=SYNTHETIC_PARITY_ATOL,
        ), f"Upstream parity failed for {label}"

    print("\nSynthetic upstream parity: PASS")

## 3A.7 — Real Core examples

Use deterministic **TRAIN-only** examples for feature visualization and distribution diagnostics.

This avoids using validation/test signal characteristics to make frontend choices.

### Select deterministic TRAIN-only examples

Choose representative training examples for visualization and real-data parity checks without using validation/test signal characteristics.

In [ ]:
train_examples = examples[examples["split"] == "train"].copy()

# One deterministic example per species + additional fixed sample.
representatives = (
    train_examples
    .sort_values(["species", "id", "start_sample_16k"])
    .groupby("species", as_index=False)
    .head(1)
)

extra = (
    train_examples
    .sort_values(["id", "start_sample_16k"])
    .iloc[::max(1, len(train_examples) // 8)]
    .head(8)
)

diagnostic_examples = (
    pd.concat([representatives, extra], ignore_index=True)
    .drop_duplicates("example_uid")
    .reset_index(drop=True)
)

display(
    diagnostic_examples[
        [
            "example_uid",
            "id",
            "species",
            "start_sample_16k",
        ]
    ]
)

print("Diagnostic examples:", len(diagnostic_examples))

### Inspect one real example

Extract one 15,600-sample context, compute its `96 × 64` Log-Mel representation, and visualize both waveform and feature tensor.

In [ ]:
row = diagnostic_examples.iloc[0]

waveform_16k = load_core_wav_16k(row["id"])
context = extract_reference_context(row, waveform_16k)
logmel = logmel_reference(context)

print("Example:", row["example_uid"])
print("Species:", row["species"])
print("Context shape:", context.shape)
print("Log-Mel shape:", logmel.shape)
print(
    "Log-Mel range:",
    float(logmel.min()),
    "to",
    float(logmel.max()),
)

assert len(context) == 15600
assert logmel.shape == (96, 64)
assert np.isfinite(logmel).all()

plt.figure(figsize=(12, 3))
plt.plot(np.arange(len(context)) / SAMPLE_RATE, context)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title(f"Reference context — {row['species']} — {row['example_uid']}")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.imshow(
    logmel.T,
    origin="lower",
    aspect="auto",
    interpolation="nearest",
)
plt.xlabel("Feature frame")
plt.ylabel("Mel bin")
plt.title("VGGish-style Log-Mel — 96 × 64")
plt.colorbar(label="ln(mel + 0.01)")
plt.tight_layout()
plt.show()

### Real Core numerical parity

Compare the project implementation against the pinned upstream frontend on real training contexts, not only synthetic signals.

In [ ]:
REAL_PARITY_ATOL = 1e-10
real_parity_results = []

if upstream is None:
    if REQUIRE_UPSTREAM_PARITY:
        raise RuntimeError("Upstream module is required for real-data parity.")
else:
    # Cache resampled clips so examples from the same clip are not
    # repeatedly decoded/resampled.
    clip_cache = {}

    for _, row in diagnostic_examples.iterrows():
        clip_id = int(row["id"])

        if clip_id not in clip_cache:
            clip_cache[clip_id] = load_core_wav_16k(clip_id)

        context = extract_reference_context(
            row,
            waveform_16k=clip_cache[clip_id],
        )

        ours = logmel_reference(context)

        theirs = upstream.log_mel_spectrogram(
            context,
            audio_sample_rate=SAMPLE_RATE,
            log_offset=LOG_OFFSET,
            window_length_secs=FRAME_LENGTH / SAMPLE_RATE,
            hop_length_secs=FRAME_HOP / SAMPLE_RATE,
            num_mel_bins=NUM_MEL_BINS,
            lower_edge_hertz=MEL_MIN_HZ,
            upper_edge_hertz=MEL_MAX_HZ,
        )

        theirs = np.asarray(theirs, dtype=np.float64)

        assert theirs.shape == EXPECTED_SHAPE

        diff = np.abs(ours - theirs)

        result = {
            "example_uid": row["example_uid"],
            "id": clip_id,
            "species": row["species"],
            "max_abs_diff": float(diff.max()),
            "mean_abs_diff": float(diff.mean()),
        }

        real_parity_results.append(result)

        assert np.allclose(
            ours,
            theirs,
            rtol=0,
            atol=REAL_PARITY_ATOL,
        ), (
            f"Real upstream parity failed: "
            f"{row['example_uid']} max={diff.max()}"
        )

    parity_df = pd.DataFrame(real_parity_results)
    display(parity_df)

    print(
        "Worst real max abs diff:",
        parity_df["max_abs_diff"].max(),
    )
    print("Real Core upstream parity: PASS")

## 3A.8 — TRAIN-only frontend audit

This is **diagnostic only**. No example is filtered based on feature magnitude or energy.

We audit a deterministic subset rather than writing all ~32k feature tensors to disk.

### Deterministic TRAIN-only feature audit

Audit a fixed subset for shape, finiteness, and basic feature statistics. These diagnostics do **not** filter examples or tune using validation/test data.

In [ ]:
AUDIT_N = min(128, len(train_examples))

# Deterministic approximately balanced sample across species.
# Avoid groupby.apply version-specific behavior.
per_species_n = max(
    1,
    AUDIT_N // train_examples["species"].nunique(),
)

audit_parts = []

for species, group in (
    train_examples
    .sort_values(["species", "id", "start_sample_16k"])
    .groupby("species", sort=True)
):
    take_n = min(per_species_n, len(group))
    positions = np.linspace(
        0,
        len(group) - 1,
        take_n,
        dtype=int,
    )
    audit_parts.append(group.iloc[positions])

audit_examples = (
    pd.concat(audit_parts, ignore_index=True)
    .drop_duplicates("example_uid")
    .head(AUDIT_N)
    .reset_index(drop=True)
)

feature_audit_records = []
clip_cache = {}

for n, (_, row) in enumerate(audit_examples.iterrows(), start=1):
    clip_id = int(row["id"])

    if clip_id not in clip_cache:
        clip_cache[clip_id] = load_core_wav_16k(clip_id)

    context = extract_reference_context(
        row,
        waveform_16k=clip_cache[clip_id],
    )
    feat = logmel_reference(context)

    feature_audit_records.append({
        "example_uid": row["example_uid"],
        "id": clip_id,
        "species": row["species"],
        "min": float(feat.min()),
        "max": float(feat.max()),
        "mean": float(feat.mean()),
        "std": float(feat.std()),
        "finite": bool(np.isfinite(feat).all()),
        "shape_ok": feat.shape == EXPECTED_SHAPE,
    })

feature_audit = pd.DataFrame(feature_audit_records)

assert feature_audit["finite"].all()
assert feature_audit["shape_ok"].all()

display(feature_audit.groupby("species")[["min", "max", "mean", "std"]].describe())

print("Audited TRAIN examples:", len(feature_audit))
print("TRAIN-only frontend diagnostic: PASS")

### Feature-distribution plots

Visualize TRAIN-only per-example Log-Mel means and standard deviations as descriptive diagnostics.

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(feature_audit["mean"], bins=25)
plt.xlabel("Per-example Log-Mel mean")
plt.ylabel("Count")
plt.title("TRAIN-only Log-Mel mean distribution")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
plt.hist(feature_audit["std"], bins=25)
plt.xlabel("Per-example Log-Mel standard deviation")
plt.ylabel("Count")
plt.title("TRAIN-only Log-Mel standard-deviation distribution")
plt.tight_layout()
plt.show()

## 3A.9 — Do not precompute the full feature dataset yet

`32,645 × 96 × 64 × float32` is roughly **0.8 GB** before labels/metadata.

There is no need to create a large feature file simply to freeze the frontend. Model training can later compute features through the verified function or a carefully designed cache.

The only file written by this notebook is a small audit/provenance JSON.

### Record artifact identities

Hash the current manifest, frozen split, and pinned upstream frontend so the verification can be reproduced later.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest_sha256 = sha256_file(MANIFEST_PATH)
split_sha256 = sha256_file(SPLIT_PATH)

print("Manifest SHA-256:", manifest_sha256)
print("Split SHA-256:", split_sha256)

if upstream_sha256 is not None:
    print("Upstream mel_features.py SHA-256:", upstream_sha256)

### Save the 03A audit record

Write only a small provenance/audit JSON. The full 32,645-example feature dataset is intentionally not materialized here.

In [ ]:
audit_dir_candidates = [
    PROJECT / "data" / "audits" / "current",
    PROJECT / "data",
]

AUDIT_DIR = next(
    (p for p in audit_dir_candidates if p.exists()),
    PROJECT / "data",
)

AUDIT_PATH = AUDIT_DIR / "03A_logmel_reference_frontend_audit.json"

worst_synthetic_diff = (
    max(r["max_abs_diff"] for r in synthetic_parity_results)
    if synthetic_parity_results else None
)

worst_real_diff = (
    max(r["max_abs_diff"] for r in real_parity_results)
    if real_parity_results else None
)

audit_report = {
    "status": "VERIFIED_IMPLEMENTATION_READY_TO_FREEZE",
    "no_model_training": True,
    "manifest": {
        "path": str(MANIFEST_PATH.relative_to(PROJECT)),
        "sha256": manifest_sha256,
        "examples": int(len(examples)),
    },
    "frozen_split": {
        "path": str(SPLIT_PATH.relative_to(PROJECT)),
        "sha256": split_sha256,
    },
    "frontend": {
        "sample_rate_hz": SAMPLE_RATE,
        "context_samples": CONTEXT_SAMPLES,
        "context_seconds": CONTEXT_SAMPLES / SAMPLE_RATE,
        "frame_length_samples": FRAME_LENGTH,
        "frame_hop_samples": FRAME_HOP,
        "fft_length": FFT_LENGTH,
        "frames": EXPECTED_FRAMES,
        "mel_bins": NUM_MEL_BINS,
        "mel_min_hz": MEL_MIN_HZ,
        "mel_max_hz": MEL_MAX_HZ,
        "log_offset": LOG_OFFSET,
        "window": "periodic_hann",
        "spectrum": "unnormalized_magnitude_rfft",
        "mel_mapping": "VGGish_HTK_style",
        "mel_area_normalization": False,
        "dc_weight_zero": True,
        "log": "natural_log",
        "output_shape": [EXPECTED_FRAMES, NUM_MEL_BINS],
    },
    "upstream_parity": {
        "required": REQUIRE_UPSTREAM_PARITY,
        "source_path": (
            str(UPSTREAM_MEL_FEATURES_PATH.relative_to(PROJECT))
            if UPSTREAM_MEL_FEATURES_PATH is not None
            and PROJECT in UPSTREAM_MEL_FEATURES_PATH.parents
            else (
                str(UPSTREAM_MEL_FEATURES_PATH)
                if UPSTREAM_MEL_FEATURES_PATH is not None
                else None
            )
        ),
        "source_sha256": upstream_sha256,
        "synthetic_cases": len(synthetic_parity_results),
        "worst_synthetic_max_abs_diff": worst_synthetic_diff,
        "real_cases": len(real_parity_results),
        "worst_real_max_abs_diff": worst_real_diff,
        "atol": max(SYNTHETIC_PARITY_ATOL, REAL_PARITY_ATOL),
    },
    "train_only_diagnostic_examples": int(len(feature_audit)),
    "software": {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "pandas": pd.__version__,
    },
}

AUDIT_PATH.write_text(
    json.dumps(audit_report, indent=2),
    encoding="utf-8",
)

print("Saved:", AUDIT_PATH)

## 3A.10 — Final gate

This notebook may be promoted from **PROPOSED** to **VERIFIED / FROZEN frontend** only if:

- Example Grid v2 authentication passes.
- `15,600 → 96` frame geometry passes.
- synthetic frontend tests pass.
- numerical parity against the pinned upstream `mel_features.py` passes.
- real Core example parity passes.
- all audited tensors are finite and exactly `96 × 64`.

Freezing this frontend does **not** freeze:

- a CNN architecture,
- training hyperparameters,
- feature standardization for a future model,
- Edge MFE,
- ESP32 deployment preprocessing.

In [ ]:
assert len(examples) == 32645
assert complete_frame_count(CONTEXT_SAMPLES, FRAME_LENGTH, FRAME_HOP) == 96
assert MEL_MATRIX.shape == (257, 64)
assert feature_audit["finite"].all()
assert feature_audit["shape_ok"].all()

if REQUIRE_UPSTREAM_PARITY:
    assert upstream is not None
    assert len(synthetic_parity_results) == len(synthetic_inputs)
    assert len(real_parity_results) == len(diagnostic_examples)

    assert worst_synthetic_diff <= SYNTHETIC_PARITY_ATOL
    assert worst_real_diff <= REAL_PARITY_ATOL

print("03A_REFERENCE_FRONTEND: PASS")
print()
print("Frontend implementation is VERIFIED and ready for methodological freeze.")
print("No classifier was trained.")
print("Next frontend task: 03B Edge MFE (separate contract).")